# NumCompute Quickstart

This notebook demonstrates an end-to-end workflow for the current `RY` branch version of `NumCompute`.

It covers the demo requirements by showing how to:

- read CSV data with `io.py`
- preprocess data with `preprocessing.py`
- chain preprocessors with `pipeline.py`
- use `sort_search.py` and `rank.py`
- compute statistics with `stats.py`
- estimate gradients and Jacobians with `optim.py`
- compare vectorised implementations against Python loops with a short benchmark summary

## 1. Setup imports

In [47]:
import sys
from pathlib import Path
import numpy as np
import time

cwd = Path.cwd()
candidate_paths = [cwd, cwd.parent]
for p in candidate_paths:
    if (p / "numcompute").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        repo_root = p
        break
else:
    raise FileNotFoundError("Could not find the repository root containing the 'numcompute' package.")

print("Repository root:", repo_root)

Repository root: c:\Users\Administrator\Desktop\works\programming-for-ai-group-assignment


In [48]:
from numcompute.io import load_csv
from numcompute.preprocessing import SimpleImputer, StandardScaler, MinMaxScaler, OneHotEncoder
from numcompute.pipeline import Pipeline
from numcompute.sort_search import SortSearch
from numcompute.rank import Rank
from numcompute.stats import Stats
from numcompute.optim import grad, jacobian

## 2. Read CSV using `io.py`

In [49]:
data_path = repo_root / "student_performance_data.csv"
X_raw = load_csv(str(data_path), delimiter=",", skip=1)

print("Raw loaded data:")
print(X_raw)
print()
print("Type:", type(X_raw))
print("Shape:", X_raw.shape)

Raw loaded data:
[['1' '19' '12.5' '0.91' '78' '74' 'CS' 'Online' '1']
 ['2' '21' '9' '0.85' '69' '65' 'IT' 'OnCampus' '1']
 ['3' '20' '15' '0.97' '88' '90' 'DS' 'Online' '1']
 ['4' '22' '6.5' '0.72' '55' '58' 'SE' 'Hybrid' '0']
 ['5' '23' '8' '0.8' '61' '63' 'CS' 'OnCampus' '0']
 ['6' '20' '11' '0.89' '73' '70' 'IT' 'Hybrid' '1']
 ['7' '24' '4.5' '0.6' '48' '52' 'DS' 'Online' '0']
 ['8' '21' '13' '0.95' '85' '87' 'CS' 'Hybrid' '1']
 ['9' '19' '7.5' '0.78' '64' '60' 'SE' 'OnCampus' '0']
 ['10' '22' '10.5' '0.88' '72' '76' 'IT' 'Online' '1']
 ['11' '20' '14' '0.96' '90' '92' 'DS' 'Hybrid' '1']
 ['12' '23' '5.5' '0.67' '51' '49' 'CS' 'OnCampus' '0']
 ['13' '21' '9.5' '0.83' '68' '71' 'SE' 'Online' '1']
 ['14' '25' '3' '0.55' '42' '45' 'IT' 'Hybrid' '0']
 ['15' '19' '16' '0.99' '93' '95' 'CS' 'OnCampus' '1']
 ['16' '22' '8.5' '0.81' '66' '64' 'DS' 'Online' '1']
 ['17' '20' 'nan' '0.9' '75' '73' 'IT' 'Hybrid' '1']
 ['18' '24' '6' 'nan' '57' '59' 'SE' 'OnCampus' '0']
 ['19' '21' '12' '0.93'

## 3. Preprocess numeric data using `preprocessing.py`

Below we create a small numeric subset containing missing values, then apply:
- `SimpleImputer`
- `StandardScaler`
- `MinMaxScaler`

In [50]:
X_numeric = np.array([
    [20.0, np.nan, 88.0],
    [21.0, 4.0, 91.0],
    [19.0, 2.8, np.nan]
], dtype=float)

print("Original numeric data:")
print(X_numeric)

Original numeric data:
[[20.   nan 88. ]
 [21.   4.  91. ]
 [19.   2.8  nan]]


In [51]:
imputer = SimpleImputer(strategy="constant", fill_value=0)
X_imputed = imputer.fit_transform(X_numeric)

print("Imputed numeric data:")
print(X_imputed)

Imputed numeric data:
[[20.   0.  88. ]
 [21.   4.  91. ]
 [19.   2.8  0. ]]


In [52]:
std_scaler = StandardScaler()
X_standard = std_scaler.fit_transform(X_imputed)

print("Standard-scaled data:")
print(X_standard)
print()
print("Means learned:", std_scaler.mean_)
print("Scales learned:", std_scaler.scale_)

Standard-scaled data:
[[ 0.         -1.35244738  0.67127116]
 [ 1.22474487  1.03422447  0.74234693]
 [-1.22474487  0.31822291 -1.41361808]]

Means learned: [20.          2.26666667 59.66666667]
Scales learned: [ 0.81649658  1.67597401 42.20847729]


In [53]:
minmax_scaler = MinMaxScaler()
X_minmax = minmax_scaler.fit_transform(X_imputed)

print("Min-max scaled data:")
print(X_minmax)
print()
print("Minimums learned:", minmax_scaler.min_)
print("Maximums learned:", minmax_scaler.max_)

Min-max scaled data:
[[0.5        0.         0.96703297]
 [1.         1.         1.        ]
 [0.         0.7        0.        ]]

Minimums learned: [19.  0.  0.]
Maximums learned: [21.  4. 91.]


## 4. Encode categorical features using `OneHotEncoder`

In [54]:
X_categorical = np.array([
    ["CS", "Online"],
    ["Math", "Offline"],
    ["CS", "Offline"],
    ["Physics", "Online"]
], dtype=object)

encoder = OneHotEncoder()
X_encoded = encoder.fit_transform(X_categorical)

print("Original categorical data:")
print(X_categorical)
print()
print("Learned categories:")
print(encoder.categories_)
print()
print("One-hot encoded data:")
print(X_encoded)

Original categorical data:
[['CS' 'Online']
 ['Math' 'Offline']
 ['CS' 'Offline']
 ['Physics' 'Online']]

Learned categories:
[array(['CS', 'Math', 'Physics'], dtype=object), array(['Offline', 'Online'], dtype=object)]

One-hot encoded data:
[[1. 0. 0. 0. 1.]
 [0. 1. 0. 1. 0.]
 [1. 0. 0. 1. 0.]
 [0. 0. 1. 0. 1.]]


## 5. Chain preprocessing steps with `Pipeline`

In [55]:
preprocess_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ("scaler", StandardScaler())
])

X_processed = preprocess_pipe.fit_transform(X_numeric)

print("Pipeline output:")
print(X_processed)

Pipeline output:
[[ 0.         -1.35244738  0.67127116]
 [ 1.22474487  1.03422447  0.74234693]
 [-1.22474487  0.31822291 -1.41361808]]


## 6. Show `sort_search` and `rank` features

This section demonstrates:
- stable sorting
- top-k selection
- quickselect / k-th element
- binary search
- ranking with ties
- percentile calculation

In [56]:
arr = np.array([7, 2, 9, 2, 5, 8, 3])

print("Original array:", arr)
print("Stable sort:", SortSearch.stable_sort(arr))
print("Top-3 indices (largest):", SortSearch.topk(arr, 3, largest=True, return_indices=True))


sorted_arr = SortSearch.stable_sort(arr)
insert_idx, exists = SortSearch.binary_search(sorted_arr, 5)
print("Binary search on sorted array -> insertion index:", insert_idx, "| exists:", exists)

print("Average rank:", Rank.rank(arr, method="average"))
print("Dense rank:", Rank.rank(arr, method="dense"))
print("Ordinal rank:", Rank.rank(arr, method="ordinal"))
print("25th percentile:", Rank.percentile(arr, 25, interpolation="linear"))

Original array: [7 2 9 2 5 8 3]
Stable sort: [2 2 3 5 7 8 9]
Top-3 indices (largest): [0 5 2]
Binary search on sorted array -> insertion index: 3 | exists: True
Average rank: [5.  1.5 7.  1.5 4.  6.  3. ]
Dense rank: [4 1 6 1 3 5 2]
Ordinal rank: [5 1 7 2 4 6 3]
25th percentile: 2.5


## 7. Compute descriptive statistics with `stats.py`

This section shows:
- mean
- median
- standard deviation
- minimum and maximum
- histogram
- quantiles

In [57]:
stats_obj = Stats(X_numeric)

print("Mean:", stats_obj.mean())
print("Median:", stats_obj.median())
print("Standard deviation:", stats_obj.std())
print("Minimum:", stats_obj.min())
print("Maximum:", stats_obj.max())

hist_counts, hist_bins = stats_obj.histogram(bins=4)
print("Histogram counts:", hist_counts)
print("Histogram bins:", hist_bins)

print("Quantiles [0.25, 0.5, 0.75]:", stats_obj.quantiles([0.25, 0.5, 0.75]))
print("Axis-wise stats (axis=0):", stats_obj.axis_stats(axis=0))

Mean: 35.114285714285714
Median: 20.0
Standard deviation: 35.09120478212774
Minimum: 2.8
Maximum: 91.0
Histogram counts: [5 0 0 2]
Histogram bins: [ 2.8  24.85 46.9  68.95 91.  ]
Quantiles [0.25, 0.5, 0.75]: [11.5 20.  54.5]
Axis-wise stats (axis=0): {'mean': array([20. ,  3.4, 89.5]), 'median': array([20. ,  3.4, 89.5]), 'std': array([0.81649658, 0.6       , 1.5       ]), 'min': array([19. ,  2.8, 88. ]), 'max': array([21.,  4., 91.])}


## 8. Estimate gradients and Jacobian with `optim.py`

This section demonstrates:
- numerical gradient for a scalar-valued function
- numerical Jacobian for a vector-valued function

In [58]:
def f_scalar(x):
    return x[0]**2 + 3*x[1]

x0 = np.array([2.0, 4.0], dtype=float)

g = grad(f_scalar, x0, method="central")
print("Gradient of scalar function at", x0, ":", g)

def f_vector(x):
    return np.array([
        x[0] + x[1],
        x[0] * x[1]
    ], dtype=float)

J = jacobian(f_vector, x0)
print("Jacobian of vector function at", x0, ":")
print(J)

Gradient of scalar function at [2. 4.] : [4. 3.]
Jacobian of vector function at [2. 4.] :
[[1. 1.]
 [4. 2.]]


## 9. Short benchmark: vectorised vs Python loops

The assignment asks for a short performance table comparing vectorised implementations against Python loops.
The following benchmark mirrors that goal with simple timing comparisons for `StandardScaler` and `SimpleImputer`.

In [59]:
def benchmark_scaler_local():
    n_rows, n_cols = 20000, 10
    data = np.random.rand(n_rows, n_cols)

    def loop_scaling(X):
        mean = np.mean(X, axis=0)
        std = np.std(X, axis=0)
        out = np.empty_like(X)
        for i in range(n_rows):
            for j in range(n_cols):
                out[i, j] = (X[i, j] - mean[j]) / (std[j] if std[j] != 0 else 1.0)
        return out

    start = time.perf_counter()
    loop_scaling(data)
    loop_time = time.perf_counter() - start

    scaler = StandardScaler()
    start = time.perf_counter()
    scaler.fit_transform(data)
    vec_time = time.perf_counter() - start

    return loop_time, vec_time

def benchmark_imputer_local():
    n_rows, n_cols = 20000, 10
    data = np.random.rand(n_rows, n_cols)
    mask = np.random.choice([True, False], size=data.shape, p=[0.1, 0.9])
    data[mask] = np.nan

    def loop_impute(X, fill_value=0.0):
        out = X.copy()
        for i in range(n_rows):
            for j in range(n_cols):
                if np.isnan(X[i, j]):
                    out[i, j] = fill_value
        return out

    start = time.perf_counter()
    loop_impute(data)
    loop_time = time.perf_counter() - start

    imputer = SimpleImputer(strategy="constant", fill_value=0.0)
    start = time.perf_counter()
    imputer.fit_transform(data)
    vec_time = time.perf_counter() - start

    return loop_time, vec_time

sc_loop, sc_vec = benchmark_scaler_local()
im_loop, im_vec = benchmark_imputer_local()

print("Short Performance Table")
print("-" * 60)
print(f"{'Task':<20}{'Loop Time (s)':<18}{'Vectorised Time (s)':<22}{'Speedup':<10}")
print("-" * 60)
print(f"{'StandardScaler':<20}{sc_loop:<18.4f}{sc_vec:<22.4f}{(sc_loop/sc_vec):<10.2f}")
print(f"{'SimpleImputer':<20}{im_loop:<18.4f}{im_vec:<22.4f}{(im_loop/im_vec):<10.2f}")

Short Performance Table
------------------------------------------------------------
Task                Loop Time (s)     Vectorised Time (s)   Speedup   
------------------------------------------------------------
StandardScaler      0.0629            0.0034                18.39     
SimpleImputer       0.1377            0.0009                150.42    


## 10. Environment Notes

Benchmark timings depend on:
- Python version
- NumPy version
- CPU and available system resources

For reproducibility, run the notebook in the same repository environment and report the local machine setup alongside the timing results.

## 11. Summary

This notebook now covers the main demo requirements:
- CSV reading with missing values
- preprocessing with `SimpleImputer`, `StandardScaler`, `MinMaxScaler`, and `OneHotEncoder`
- sorting, searching, ranking, and percentile examples
- descriptive statistics
- gradient and Jacobian estimation
- a short benchmark comparing vectorised operations to Python loops